# Ensemble Voting
Diese Notebook demonstriert den Aufbau und die Verwendung eines Ensemble Voting Ansatzes und die Gegenüberstellung der Aussagekraft gegenüber dem reinen RGB-Modell

In [1]:
import sys
from pathlib import Path
import pandas as pd

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

Project directory set to: E:\Admin\Documents\dermascan


In [2]:
from dataset_manager import DatasetManager
from ensemble_voting.voter_models import ResNetRGBModel, ResNetFilterModel
from voter import EnsembleVoter
from ensemble_voting.ensemble_analyser import EnsembleAnalyser

In [3]:
dataset_manager = DatasetManager(data_dir=str(proj_dir / "data"))

# Load metadata (filtering for Melanoma vs Nevi as per trainer restrictions)
dataset_manager.load_cleaned_metadata(filter_melanoma_only=True)

test_df = pd.read_csv(proj_dir / "data" / "test_df.csv")

plot_output_dir = proj_dir / "python" / "results" / "plots"
plot_output_dir.mkdir(parents=True, exist_ok=True)

label_names = ["nv", "mel"]

In [4]:
negative_count = test_df["lesion_idx"].value_counts().get(0) # Count of negative samples (label 0 == "nv")
positive_count = test_df["lesion_idx"].value_counts().get(1) # Count of positive samples (label 1 == "mel")

print(f"Negative samples (nv): {negative_count}")
print(f"Positive samples (mel): {positive_count}")
print(f"Sample distribution: {negative_count / (negative_count + positive_count):.2%} nv, {positive_count / (negative_count + positive_count):.2%} mel")


Negative samples (nv): 1006
Positive samples (mel): 167
Sample distribution: 85.76% nv, 14.24% mel


Definition von Grundmodellen, zu welchen trainierte Gewichte erstellt wurden:

In [5]:
specs = [
    ResNetRGBModel(
        model_id="resnet18_rgb",
        model_path=str(proj_dir / "python/results/models/ResNet_25022026-195039_rgb.pt")
    ),
    ResNetFilterModel(
        model_id="resnet18_gabor", 
        proj_dir=proj_dir,
        model_path=str(proj_dir / "python/results/models/ResNet_06032026-181320_gabor_filter.pt"),
        filter_name="gabor_filter",
        input_channels=40
    ),
    ResNetFilterModel(
        model_id="resnet18_lbp", 
        proj_dir=proj_dir,
        model_path=str(proj_dir / "python/results/models/ResNet_15032026-153758_lbp_filter.pt"),
        filter_name="lbp_filter",
        input_channels=1
    ),
    ResNetFilterModel(
        model_id="resnet18_glcm", 
        proj_dir=proj_dir,
        model_path=str(proj_dir / "python/results/models/ResNet_16032026-115755_glcm_filter.pt"),
        filter_name="glcm_filter",
        input_channels=32
    ),
]

### Beispielnutzung der EnsembleVoter Klasse

Soft Voting:

In [6]:
# Soft voting with equal weights for all four models (RGB, Gabor, LBP, GLCM)
voter = EnsembleVoter(
    model_specs=specs,
    strategy="soft",
    weights={"resnet18_rgb": 1, "resnet18_gabor": 1, "resnet18_lbp": 1, "resnet18_glcm": 1},
    tie_breaker="max_prob",
    label_names=label_names,
)

# Grab the first row
test_image_row = dataset_manager.df.iloc[0]

print(f"Testing on image: {test_image_row['image_id']} (Real Label: {test_image_row['lesion_idx']})")

result = voter.run(image=test_image_row)
print(f"\nSoft Voting Result: Predicted '{result.label}' with confidence {result.confidence:.4f}")

Testing on image: ISIC_0024698 (Real Label: 0)
Ensemble voting summary
Strategy: soft
Model           | Predicted       | Match? | Confidence | Prob(Final)
------------------------------------------------------------------
resnet18_rgb    | nv              | Yes    | 0.9539     | 0.9539     
resnet18_gabor  | nv              | Yes    | 0.5844     | 0.5844     
resnet18_lbp    | nv              | Yes    | 0.9293     | 0.9293     
resnet18_glcm   | nv              | Yes    | 0.8743     | 0.8743     
------------------------------------------------------------------
Ensemble        | nv              | Yes    | 0.8355     | 0.8355     

Soft Voting Result: Predicted '0' with confidence 0.8355


Hard Voting:

In [7]:
# Hard voting with a tie breaker
voter_hard = EnsembleVoter(
    model_specs=specs,
    strategy="hard",
    weights={"resnet18_rgb": 1, "resnet18_gabor": 1, "resnet18_lbp": 1, "resnet18_glcm": 1},
    tie_breaker="max_prob",
    label_names=label_names,
)

result_hard = voter_hard.run(image=test_image_row)
print(f"\nHard Voting Result: Predicted '{result_hard.label}' with confidence {result_hard.confidence:.4f}")

Ensemble voting summary
Strategy: hard
Model           | Predicted       | Match? | Confidence | Prob(Final)
------------------------------------------------------------------
resnet18_rgb    | nv              | Yes    | 0.9539     | 0.9539     
resnet18_gabor  | nv              | Yes    | 0.5844     | 0.5844     
resnet18_lbp    | nv              | Yes    | 0.9293     | 0.9293     
resnet18_glcm   | nv              | Yes    | 0.8743     | 0.8743     
------------------------------------------------------------------
Ensemble        | nv              | Yes    | 1.0000     | 1.0000     

Hard Voting Result: Predicted '0' with confidence 1.0000


## Analyse

Im Folgenden wird die EnsembleAnalyser Klasse dafür genutzt, um die Performance der unterschiedlichen Modell-Ensembles zu vergleichen.

Dazu können die o.g. Basis Modelle aufgelistet und ggf. gewichtet werden. Anschließend wird für jedes Modell durch Inferenz auf dem Test-Datensatz (`test_df`) Performance-Metriken gesammelt und weitere Artefakte berechnet.
Für jedes Modell wird die Performance durch eine Grafik mit ROC-Kurve, PR-Kurve und Verwechslungsmatrix zusammengefasst abgespeichert.

In [8]:
# Build standalone voters (single model specs) + ensemble voters
voters_to_compare = {
    "rgb_standalone": EnsembleVoter(
        model_specs=[specs[0]],
        strategy="soft",
        weights={"resnet18_rgb": 1},
        label_names=label_names,
        verbose=False,
    ),
    "gabor_filtered_standalone": EnsembleVoter(
        model_specs=[specs[1]],
        strategy="soft",
        weights={"resnet18_gabor": 1},
        label_names=label_names,
        verbose=False,
    ),
    "lbp_filtered_standalone": EnsembleVoter(
        model_specs=[specs[2]],
        strategy="soft",
        weights={"resnet18_lbp": 1},
        label_names=label_names,
        verbose=False,
    ),
    "glcm_filtered_standalone": EnsembleVoter(
        model_specs=[specs[3]],
        strategy="soft",
        weights={"resnet18_glcm": 1},
        label_names=label_names,
        verbose=False,
    ),
    "ensemble_all_soft": EnsembleVoter(
        model_specs=specs,
        strategy="soft",
        weights={
            "resnet18_rgb": 1.0,
            "resnet18_gabor": 1.0,
            "resnet18_lbp": 1.0,
            "resnet18_glcm": 1.0,
        },
        tie_breaker="max_prob",
        label_names=label_names,
        verbose=False,
    ),
    "ensemble_all_hard": EnsembleVoter(
        model_specs=specs,
        strategy="hard",
        weights={
            "resnet18_rgb": 1.0,
            "resnet18_gabor": 1.0,
            "resnet18_lbp": 1.0,
            "resnet18_glcm": 1.0,
        },
        tie_breaker="max_prob",
        label_names=label_names,
        verbose=False,
    ),
    "ensemble_gabor_rgb_soft": EnsembleVoter(
        model_specs=[specs[0], specs[1]],
        strategy="soft",
        weights={
            "resnet18_rgb": 1.0,
            "resnet18_gabor": 1.0,
        },
        tie_breaker="max_prob",
        label_names=label_names,
        verbose=False,
    ),
    "ensemble_glcm_rgb_soft": EnsembleVoter(
        model_specs=[specs[0], specs[3]],
        strategy="soft",
        weights={
            "resnet18_rgb": 1.0,
            "resnet18_glcm": 1.0,
        },
        tie_breaker="max_prob",
        label_names=label_names,
        verbose=False,
    ),
    "ensemble_lbp_rgb_soft": EnsembleVoter(
        model_specs=[specs[0], specs[2]],
        strategy="soft",
        weights={
            "resnet18_rgb": 1.0,
            "resnet18_lbp": 1.0,
        },
        tie_breaker="max_prob",
        label_names=label_names,
        verbose=False,
    ),
}

In [9]:
analyser = EnsembleAnalyser(label_column="lesion_idx", labels = label_names)
analysis_results = analyser.compare_voters(
    voters=voters_to_compare,
    data_df=test_df,
    output_dir=plot_output_dir,
)

Evaluating model 1/9: rgb_standalone
Evaluating model 2/9: gabor_filtered_standalone
Evaluating model 3/9: lbp_filtered_standalone
Evaluating model 4/9: glcm_filtered_standalone
Evaluating model 5/9: ensemble_all_soft
Evaluating model 6/9: ensemble_all_hard
Evaluating model 7/9: ensemble_gabor_rgb_soft
Evaluating model 8/9: ensemble_glcm_rgb_soft
Evaluating model 9/9: ensemble_lbp_rgb_soft


In [13]:
summary_df = analysis_results["summary_df"].sort_values(by="sensitivity", ascending=False)
display(summary_df)

print(f"Saved per-model diagnostics to: {plot_output_dir}")
print("Generated:")
print(" - model_<name>_diagnostics.png (one per model)")

,roc_auc,average_precision,sensitivity,specificity,accuracy,balanced_accuracy,precision,f1_binary,f1_macro,tn,fp,fn,tp,model_name
6,0.930007,0.711585,0.592814,0.966203,0.913043,0.779509,0.744361,0.660000,0.805073,972,34,68,99,ensemble_gabor_rgb_soft
1,0.924001,0.714415,0.574850,0.954274,0.900256,0.764562,0.676056,0.621359,0.781961,960,46,71,96,gabor_filtered_standalone
0,0.906543,0.603963,0.449102,0.963221,0.890026,0.706161,0.669643,0.537634,0.737613,969,37,92,75,rgb_standalone
4,0.923066,0.669471,0.383234,0.976143,0.891731,0.679688,0.727273,0.501961,0.720612,982,24,103,64,ensemble_all_soft
5,0.829252,0.527657,0.365269,0.975149,0.888321,0.670209,0.709302,0.482213,0.709812,981,25,106,61,ensemble_all_hard
8,0.895781,0.580958,0.359281,0.972167,0.884910,0.665724,0.681818,0.470588,0.703013,978,28,107,60,ensemble_lbp_rgb_soft
7,0.909644,0.606607,0.323353,0.972167,0.879795,0.647760,0.658537,0.433735,0.683248,978,28,113,54,ensemble_glcm_rgb_soft
2,0.825901,0.461913,0.293413,0.959245,0.864450,0.626329,0.544444,0.381323,0.652605,965,41,118,49,lbp_filtered_standalone
3,0.858525,0.515246,0.251497,0.976143,0.872975,0.613820,0.636364,0.360515,0.645000,982,24,125,42,glcm_filtered_standalone


Saved per-model diagnostics to: E:\Admin\Documents\dermascan\python\results\plots
Generated:
 - model_<name>_diagnostics.png (one per model)


Wenn der `sensitivity`-Wert als Anhaltspunkt für generelle Modellperformance genommen wird, dann ist das ResNet18 Ensemble aus RGB- und Gabor-trainierten Bildern das effektivste Modell in der Erkennung von Hautläsionen. Gerade im Bezug auf die Sensitivität der Top-3 Modelle sind auch die Standalone Varianten jeweils sehr performant und liefern gerade bei einer in diesem Kontext hohen Wichtigkeit der geringen Anzahl an Falsch-Negativen vielversprechende Ergebnisse. 

Weitere Experimente der Kombination `gabor+rgb` mit gewichtetem Einfluss.

In [11]:
further_voters = {
    "base": EnsembleVoter(
        model_specs=[specs[0], specs[1]],
        strategy="soft",
        weights={
            "resnet18_rgb": 1.0,
            "resnet18_gabor": 1.0,
        },
        tie_breaker="max_prob",
        label_names=label_names,
        verbose=False,
    ),
    ".75_gabor": EnsembleVoter(
        model_specs=[specs[0], specs[1]],
        strategy="soft",
        weights={
            "resnet18_rgb": 0.25,
            "resnet18_gabor": 0.75,
        },
        tie_breaker="max_prob",
        label_names=label_names,
        verbose=False,
    ),
    ".25_gabor": EnsembleVoter(
        model_specs=[specs[0], specs[1]],
        strategy="soft",
        weights={
            "resnet18_rgb": 0.75,
            "resnet18_gabor": 0.25,
        },
        tie_breaker="max_prob",
        label_names=label_names,
        verbose=False,
    ),
}

In [14]:
further_analysis_results = analyser.compare_voters(
    voters=further_voters,
    data_df=test_df,
    # output_dir=plot_output_dir,
)
summary_df = further_analysis_results["summary_df"].sort_values(by="sensitivity", ascending=False)
display(summary_df)

Evaluating model 1/3: base
Evaluating model 2/3: .75_gabor
Evaluating model 3/3: .25_gabor


,roc_auc,average_precision,sensitivity,specificity,accuracy,balanced_accuracy,precision,f1_binary,f1_macro,tn,fp,fn,tp,model_name
1,0.929900,0.720266,0.598802,0.962227,0.910486,0.780515,0.724638,0.655738,0.802146,968,38,67,100,.75_gabor
0,0.930007,0.711585,0.592814,0.966203,0.913043,0.779509,0.744361,0.660000,0.805073,972,34,68,99,base
2,0.923162,0.672614,0.491018,0.965209,0.897698,0.728113,0.700855,0.577465,0.759634,971,35,85,82,.25_gabor


Die Neugewichtung der Modelle im Ensemble führt bei einer 75%-Gewichtung des Gabor-Modells zu einer minimalen Leistungssteigerung, allerdings auf Kosten einer höheren Anzahl falsch-positiver Vorhersagen. Da dies im medizinischen Kontext jedoch lediglich bedeutet, dass eine Läsion fälschlicherweise als auffällig eingestuft wird – was durch weitere ärztliche Untersuchungen ohnehin validiert werden muss –, kann dieses Modell als das bisherige Top-Modell betrachtet werden. Eine höhere Falsch-Negativ-Rate wäre wesentlich kritischer, da bösartige Hautläsionen dadurch übersehen werden könnten.